# SupplyMind AI — Model Comparison & Champion Selection

Compare all candidate models on the validation partition, select the champion
according to the RFC, then evaluate that model once on the untouched latest
test partition.

**Selection order:** F1 → recall → ROC-AUC.

In [ ]:
# -------------------
# Imports
# -------------------

import json
from pathlib import Path

import pandas as pd

In [ ]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [ ]:
# -------------------
# Load candidate metrics
# -------------------

candidate_names = [
    "logistic_regression",
    "random_forest",
    "xgboost",
    "hist_gradient_boosting",
]

rows = []

for name in candidate_names:
    metrics_path = (
        REPORT_ROOT / "models" / name / "validation_metrics.json"
    )

    if not metrics_path.exists():
        print(f"Missing metrics for {name}: run its notebook first.")
        continue

    metrics = json.loads(metrics_path.read_text())
    rows.append({"model_name": name, **metrics})

comparison = pd.DataFrame(rows).sort_values(
    ["f1", "recall", "roc_auc"],
    ascending=[False, False, False],
)

comparison

In [ ]:
# -------------------
# Champion candidate
# -------------------

if comparison.empty:
    raise RuntimeError("No model results were found.")

champion_name = comparison.iloc[0]["model_name"]
champion_threshold = comparison.iloc[0]["threshold"]

print("Selected champion:", champion_name)
print("Threshold:", champion_threshold)

## Final test evaluation

For reproducibility, run:

```bash
python scripts/train_models.py
```

The script repeats the same selection, refits the selected estimator on
train + validation, evaluates once on the untouched test partition, and saves:

- `models/champion/model.joblib`
- `models/champion/metadata.json`
- final test metrics
- confusion matrix
- ROC curve
- precision–recall curve
- feature importance